# EDA — EPE: consumo mensal de energia (bronze -> silver)

Tratamento da base da **EPE** apenas. As outras duas bases do projeto tem notebook proprio:

**Papel da EPE no projeto:** e o **alvo**. Consumo mensal por UF, subsistema eletrico,
classe de consumo e tipo de contrato, de 2015 até o mês corrente.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

In [6]:
ROOT = Path.cwd().parent
BRONZE_DIR = ROOT / "data" / "bronze" / "epe_consumo"
SILVER_DIR = ROOT / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# 2015-2025 cobre El Nino 2015/16 (muito forte), 2018/19 (fraco) e 2023/24 (forte),
# com a La Nina tripla de 2020-23 como contraste.
START, END = "2015-01-01", "2025-12-31"
N_MESES = 132

## 1. Bronze — ingestao

In [7]:
import requests

URL = ("https://www.epe.gov.br/sites-pt/publicacoes-dados-abertos/"
       "dados-abertos/Documents/Dados_abertos_Consumo_Mensal.xlsx")

BRONZE_DIR.mkdir(parents=True, exist_ok=True)
path_epe = BRONZE_DIR / "consumo_mensal_historico.xlsx"

if path_epe.exists():
    print(f"ja existe: {path_epe.name} ({path_epe.stat().st_size / 1e6:.1f} MB)")
else:
    resp = requests.get(URL, headers={"User-Agent": "DataIngestionPipeline/1.0"}, timeout=120)
    resp.raise_for_status()
    path_epe.write_bytes(resp.content)
    print(f"salvo: {path_epe.name} ({len(resp.content) / 1e6:.1f} MB)")

ja existe: consumo_mensal_historico.xlsx (13.8 MB)


### 1.1 Escolha da Página

A planilha tem 10 abas. Duas trazem consumo; as demais sao recortes do setor industrial
ou series historicas antigas, descontinuadas.

In [8]:
xls = pd.ExcelFile(path_epe)
for nome in xls.sheet_names:
    print(f"  {nome}")

  CONSUMO E NUMCONS SAM
  CONSUMO E NUMCONS SAM UF
  SETOR INDUSTRIAL POR RG
  SETOR INDUSTRIAL POR UF
  CONSUMO_BEN_RG_1970-1989
  CONSUMO_ELETROBRAS_1990-2003
  ANALISE CONS NUMCONS SAM
  ANALISE CONS NUMCONS SAM UF
  ANALISE SETOR IND POR RG
  ANALISE SETOR IND POR UF


In [9]:
# `CONSUMO E NUMCONS SAM UF` e a versao com quebra por UF; `CONSUMO E NUMCONS SAM` e a
# mesma informacao ja agregada por subsistema. Usar a granular e agregar quando precisar.
raw = pd.read_excel(path_epe, sheet_name="CONSUMO E NUMCONS SAM UF")
print("shape:", raw.shape)
raw.head(10)

shape: (60935, 10)


,Data,DataExcel,UF,Regiao,Sistema,Classe,TipoConsumidor,Consumo,Consumidores,DataVersao
0,20260601,2026-06-01,AC,Norte,SISTEMAS ISOLADOS,Comercial,Cativo,187.141,482,2026-08-24
1,20260601,2026-06-01,AC,Norte,SISTEMAS ISOLADOS,Industrial,Cativo,2.678,3,2026-08-24
2,20260601,2026-06-01,AC,Norte,SISTEMAS ISOLADOS,Outros,Cativo,525.785,392,2026-08-24
3,20260601,2026-06-01,AC,Norte,SISTEMAS ISOLADOS,Residencial,Cativo,1327.189,9357,2026-08-24
4,20260601,2026-06-01,AC,Norte,SISTEMAS ISOLADOS,Rural,Cativo,1238.375,123,2026-08-24
5,20260601,2026-06-01,AC,Norte,SUDESTE / CENTRO - OESTE,Comercial,Cativo,15655.119,21845,2026-08-24
6,20260601,2026-06-01,AC,Norte,SUDESTE / CENTRO - OESTE,Comercial,Livre,7323.900,141,2026-08-24
7,20260601,2026-06-01,AC,Norte,SUDESTE / CENTRO - OESTE,Industrial,Cativo,1294.721,383,2026-08-24
8,20260601,2026-06-01,AC,Norte,SUDESTE / CENTRO - OESTE,Industrial,Livre,3539.363,49,2026-08-24
9,20260601,2026-06-01,AC,Norte,SUDESTE / CENTRO - OESTE,Outros,Cativo,16566.926,4082,2026-08-24


---
## 2. Exploracao

### 2.1 Perfil geral

In [10]:
print(raw.dtypes.to_string(), "\n")
print("nulos por coluna:\n", raw.isna().sum().to_string(), "\n")
print("memoria:", f"{raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Data                       int64
DataExcel         datetime64[us]
UF                           str
Regiao                       str
Sistema                      str
Classe                       str
TipoConsumidor               str
Consumo                  float64
Consumidores               int64
DataVersao        datetime64[us] 

nulos por coluna:
 Data              0
DataExcel         0
UF                0
Regiao            0
Sistema           0
Classe            0
TipoConsumidor    0
Consumo           0
Consumidores      0
DataVersao        0 

memoria: 19.7 MB


In [11]:
# `Data` vem como INTEIRO no formato AAAAMMDD — nao e datetime nem string.
print("tipo de Data:", raw["Data"].dtype, "| amostra:", raw["Data"].head(3).tolist())

raw["data"] = pd.to_datetime(raw["Data"], format="%Y%m%d")
print("cobertura:", raw.data.min().date(), "->", raw.data.max().date())
print("meses distintos:", raw.data.nunique())

# A serie se estende alem do periodo de analise nos dois sentidos.
print(f"\nlinhas dentro de {START[:4]}-{END[:4]}:",
      raw.data.between(START, END).sum(), "de", len(raw))

tipo de Data: int64 | amostra: [20260601, 20260601, 20260601]
cobertura: 2004-01-01 -> 2026-06-01
meses distintos: 270

linhas dentro de 2015-2025: 32876 de 60935


### 2.2 As dimensões

In [12]:
for col in ["Regiao", "Sistema", "Classe", "TipoConsumidor", "UF"]:
    vals = raw[col].value_counts()
    print(f"--- {col}: {len(vals)} valores")
    print(vals.head(8).to_string(), "\n")

--- Regiao: 5 valores
Regiao
Nordeste        19550
Norte           16829
Centro-Oeste     9475
Sudeste          8691
Sul              6390 

--- Sistema: 5 valores
Sistema
SUDESTE / CENTRO - OESTE    19592
NORDESTE                    15770
SISTEMAS ISOLADOS           11275
NORTE INTERLIGADO            7908
SUL                          6390 

--- Classe: 5 valores
Classe
Industrial     15096
Comercial      14189
Outros         11354
Rural          11127
Residencial     9169 

--- TipoConsumidor: 2 valores
TipoConsumidor
Cativo    44321
Livre     16614 

--- UF: 27 valores
UF
PE    3423
PA    3407
MT    3217
RO    2746
AC    2697
AM    2531
RJ    2351
SP    2330 



In [13]:
# `Sistema` e o subsistema ELETRICO e nao acompanha a geografia: o Acre, Rondonia,
# Mato Grosso e Mato Grosso do Sul aparecem em SUDESTE / CENTRO - OESTE.
print(pd.crosstab(raw["Regiao"], raw["Sistema"]))

Sistema       NORDESTE  NORTE INTERLIGADO  SISTEMAS ISOLADOS  SUDESTE / CENTRO - OESTE   SUL
Regiao                                                                                      
Centro-Oeste         0                  0               1245                      8230     0
Nordeste         15770               2006               1774                         0     0
Norte                0               5902               8256                      2671     0
Sudeste              0                  0                  0                      8691     0
Sul                  0                  0                  0                         0  6390


#### A chave primaria inclui `Sistema`

Onze UFs sao **divididas** entre dois subsistemas — parte do territorio em sistema
isolado, parte interligada. Tratar `(Data, UF, Classe, TipoConsumidor)` como chave
colapsaria essas linhas.

In [14]:
divididas = raw.groupby("UF")["Sistema"].nunique().loc[lambda s: s > 1]
print("UFs em mais de um subsistema:", divididas.index.tolist(), "\n")

chave = ["Data", "UF", "Sistema", "Classe", "TipoConsumidor"]
print("duplicatas na chave completa:    ", raw.duplicated(chave).sum())
print("duplicatas sem `Sistema`:        ", raw.duplicated([c for c in chave if c != "Sistema"]).sum())

print("\nexemplo — Acre, 2016-09:")
print(raw.query("Data == 20160901 and UF == 'AC'")
         .loc[:, ["UF", "Regiao", "Sistema", "Classe", "TipoConsumidor", "Consumo"]]
         .head(8).to_string(index=False))

UFs em mais de um subsistema: ['AC', 'AM', 'AP', 'BA', 'MA', 'MS', 'MT', 'PA', 'PE', 'RO', 'RR'] 

duplicatas na chave completa:     0
duplicatas sem `Sistema`:         7998

exemplo — Acre, 2016-09:
UF Regiao                  Sistema      Classe TipoConsumidor  Consumo
AC  Norte        SISTEMAS ISOLADOS   Comercial         Cativo   2362.0
AC  Norte        SISTEMAS ISOLADOS  Industrial         Cativo    258.0
AC  Norte        SISTEMAS ISOLADOS      Outros         Cativo   3513.0
AC  Norte        SISTEMAS ISOLADOS Residencial         Cativo   7369.0
AC  Norte        SISTEMAS ISOLADOS       Rural         Cativo   1571.0
AC  Norte SUDESTE / CENTRO - OESTE   Comercial         Cativo  16314.0
AC  Norte SUDESTE / CENTRO - OESTE   Comercial          Livre   1911.0
AC  Norte SUDESTE / CENTRO - OESTE  Industrial         Cativo   2523.0


### 2.3 Qualidade dos valores

In [15]:
janela = raw[raw.data.between(START, END)]

neg = janela[janela["Consumo"] < 0]
print(f"consumo negativo:   {len(neg)} linhas ({100 * len(neg) / len(janela):.3f}%), "
      f"somando {neg['Consumo'].sum():,.0f} MWh contra {janela['Consumo'].sum():,.0f} MWh totais")
print(f"consumo zerado:     {(janela['Consumo'] == 0).sum()}")
print(f"consumidores <= 0:  {(janela['Consumidores'] <= 0).sum()}")

print("\nmaiores negativos:")
print(neg.nsmallest(5, "Consumo")
         .loc[:, ["data", "UF", "Sistema", "Classe", "TipoConsumidor", "Consumo", "Consumidores"]]
         .to_string(index=False))

consumo negativo:   8 linhas (0.024%), somando -6,141 MWh contra 5,505,599,515 MWh totais
consumo zerado:     42
consumidores <= 0:  28

maiores negativos:
      data UF                  Sistema      Classe TipoConsumidor   Consumo  Consumidores
2025-01-01 DF SUDESTE / CENTRO - OESTE  Industrial         Cativo -4093.602           932
2024-03-01 SP SUDESTE / CENTRO - OESTE Residencial          Livre  -932.115             3
2023-09-01 AP        SISTEMAS ISOLADOS  Industrial         Cativo  -287.709             5
2024-12-01 RO        SISTEMAS ISOLADOS   Comercial         Cativo  -238.431          -232
2017-10-01 AM        NORTE INTERLIGADO       Rural         Cativo  -230.000           935


Negativos sao **refaturamentos retroativos** que, no mes em que caem, superam o consumo
do periodo. Sao ruido da metodologia da EPE, nao erro de leitura. O que importa e a
materialidade — uma parte em ~1,6 milhao — e nao a contagem, entao ficam no silver.

Ha tambem uma linha com **numero de consumidores negativo**, que uma checagem de `== 0`
deixaria passar e produziria uma razao de sinal invertido mais adiante.

In [16]:
# Unidade: a planilha nao declara. Conferir por ordem de grandeza.
amostra = janela.query(
    "Data == 20160901 and UF == 'MG' and Classe == 'Comercial' and TipoConsumidor == 'Cativo'"
).iloc[0]
print(f"MG / Comercial / Cativo em 2016-09")
print(f"  consumo:      {amostra['Consumo']:,.0f}")
print(f"  consumidores: {amostra['Consumidores']:,.0f}")
print(f"  razao:        {amostra['Consumo'] / amostra['Consumidores']:.3f}")
print(f"\nSe a unidade for MWh -> {amostra['Consumo'] * 1000 / amostra['Consumidores']:,.0f} kWh/mes")
print("por unidade comercial, que e plausivel. Se fosse kWh daria 0.6 kWh/mes; se fosse")
print("GWh daria 640 MWh/mes. A unidade e MWh.")

MG / Comercial / Cativo em 2016-09
  consumo:      489,254
  consumidores: 764,547
  razao:        0.640

Se a unidade for MWh -> 640 kWh/mes
por unidade comercial, que e plausivel. Se fosse kWh daria 0.6 kWh/mes; se fosse
GWh daria 640 MWh/mes. A unidade e MWh.


---
## 3. Silver

Grao: **mes x UF x sistema x classe x tipo_consumidor**, recortado em 2015-2025.

| Coluna | Origem |
|---|---|
| `data` | `Data` (inteiro `AAAAMMDD`) convertida |
| `uf` | `UF` |
| `regiao` | `Regiao` — dimensao **geografica**, chave do join com o clima |
| `sistema` | `Sistema` — subsistema **eletrico**, chave do canal hidrico |
| `sistema_isolado` | `Sistema == "SISTEMAS ISOLADOS"` |
| `classe` | `Classe` |
| `tipo_consumidor` | `TipoConsumidor` |
| `consumo_mwh` | `Consumo` |
| `consumidores` | `Consumidores` |
| `consumo_por_consumidor` | derivada — **e este o alvo** |

Descartadas: `DataExcel` (serial do Excel, duplica `Data`) e `DataVersao` (constante).

`regiao` e `sistema` ficam as duas: temperatura e regional, reservatorio e por subsistema.

In [17]:
epe = (
    raw.query("@START <= data <= @END")
    .assign(sistema_isolado=lambda d: d["Sistema"].eq("SISTEMAS ISOLADOS"))
    .rename(columns={
        "UF": "uf",
        "Regiao": "regiao",
        "Sistema": "sistema",
        "Classe": "classe",
        "TipoConsumidor": "tipo_consumidor",
        "Consumo": "consumo_mwh",
        "Consumidores": "consumidores",
    })
    .loc[:, ["data", "uf", "regiao", "sistema", "sistema_isolado", "classe",
             "tipo_consumidor", "consumo_mwh", "consumidores"]]
)

# Divisao protegida. `> 0` e nao `!= 0`: ha uma linha com -232 consumidores, que passaria
# por uma checagem de zero e produziria uma razao de sinal invertido.
epe["consumo_por_consumidor"] = epe["consumo_mwh"] / epe["consumidores"].where(epe["consumidores"] > 0)

epe = epe.sort_values(["data", "uf", "sistema", "classe", "tipo_consumidor"]).reset_index(drop=True)

destino = SILVER_DIR / "epe_consumo.csv"
epe.to_csv(destino, index=False)
print(f"{destino.name}: {epe.shape[0]:,} linhas x {epe.shape[1]} colunas "
      f"({destino.stat().st_size / 1024:,.0f} KB)")
epe.head()

epe_consumo.csv: 32,876 linhas x 10 colunas (3,016 KB)


,data,uf,regiao,sistema,sistema_isolado,classe,tipo_consumidor,consumo_mwh,consumidores,consumo_por_consumidor
0,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Comercial,Cativo,2101.0,4376,0.480119
1,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Industrial,Cativo,274.0,172,1.593023
2,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Outros,Cativo,3413.0,1248,2.734776
3,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Residencial,Cativo,6497.0,48170,0.134876
4,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Rural,Cativo,1141.0,5826,0.195846


---
## 4. Validacao

In [18]:
# --- estrutura
assert epe["data"].dt.to_period("M").nunique() == N_MESES, "meses faltando"
assert not epe.duplicated(["data", "uf", "sistema", "classe", "tipo_consumidor"]).any(), "chave duplicada"
assert epe[["data", "uf", "regiao", "sistema", "classe", "tipo_consumidor"]].notna().all().all(), "dimensao nula"
assert epe["regiao"].nunique() == 5 and epe["classe"].nunique() == 5, "cardinalidade inesperada"

# Silver preserva o dado como ele e, entao negativos de refaturamento ficam. O que nao
# podem e virar material: se passarem de 0.1% do total, algo mudou na fonte.
_neg = epe.loc[epe["consumo_mwh"] < 0, "consumo_mwh"].sum()
assert abs(_neg) < 0.001 * epe["consumo_mwh"].sum(), f"negativos materiais ({_neg:,.0f} MWh)"

# A divisao protegida nao pode ter vazado inf.
assert np.isfinite(epe["consumo_por_consumidor"].dropna()).all(), "divisao por zero vazou"

print("estrutura OK")

estrutura OK


### 4.1 Sazonalidade — o teste de que `Data` foi lida certo

Se o parse do inteiro `AAAAMMDD` estivesse errado, o pico residencial cairia num mes
arbitrario em vez do verao.

A razao tem que ser calculada sobre os **agregados** (soma do consumo dividida pela soma
dos consumidores), nao como media das razoes linha a linha: esta ultima da o mesmo peso a
Sao Paulo e a um sistema isolado do Acre, e desloca o pico.

In [19]:
resid_val = epe.query("classe == 'Residencial' and tipo_consumidor == 'Cativo'")


def por_mes(g):
    a = g.groupby(g["data"].dt.month).agg(c=("consumo_mwh", "sum"), n=("consumidores", "sum"))
    return a["c"] / a["n"]


nacional = por_mes(resid_val)
print("nacional, consumo residencial por consumidor (MWh/mes):")
print(nacional.round(4).to_string())

assert nacional.idxmax() in (1, 2, 3, 12), f"pico no mes {nacional.idxmax()}, esperado verao"
print(f"\npico no mes {nacional.idxmax()} -> OK")

nacional, consumo residencial por consumidor (MWh/mes):
data
1     0.1784
2     0.1741
3     0.1775
4     0.1703
5     0.1638
6     0.1577
7     0.1553
8     0.1568
9     0.1628
10    0.1683
11    0.1697
12    0.1739

pico no mes 1 -> OK


#### O Norte tem sazonalidade invertida

Vale registrar antes de modelar: o pico nao cai no verao em todas as regioes.

In [20]:
for regiao, g in resid_val.groupby("regiao"):
    s = por_mes(g)
    print(f"{regiao:14s} pico: mes {s.idxmax():2d}   vale: mes {s.idxmin():2d}   "
          f"amplitude: {100 * (s.max() / s.min() - 1):4.1f}%")

Centro-Oeste   pico: mes 10   vale: mes  7   amplitude: 24.1%
Nordeste       pico: mes 12   vale: mes  7   amplitude: 13.7%
Norte          pico: mes 10   vale: mes  2   amplitude: 20.3%
Sudeste        pico: mes  3   vale: mes  7   amplitude: 18.4%
Sul            pico: mes  2   vale: mes  9   amplitude: 21.7%


Sudeste e Sul picam em fev-mar, Nordeste em dezembro — refrigeracao no verao austral.
O **Norte pica em outubro e tem o vale em fevereiro**: e o verao amazonico, a estacao seca
e quente de set-nov, enquanto jan-mar e o periodo chuvoso e mais ameno. O Centro-Oeste
mostra o mesmo deslocamento, mais fraco.

Consequencia para a gold: um modelo nacional unico soma dois ciclos de fase oposta e
perde os dois. A interacao com `regiao` — ou um modelo por regiao — nao e refinamento
opcional, e requisito. E a mesma razao pela qual o clima e coletado em cinco capitais e
nao numa so.

In [21]:
# --- cobertura: toda combinacao regiao x classe presente em todos os meses.
cob = epe.groupby(["regiao", "classe"])["data"].nunique()
assert cob.eq(N_MESES).all(), f"cobertura incompleta:\n{cob[cob != N_MESES]}"
print(f"25 combinacoes regiao x classe, {N_MESES} meses cada -> OK")

print(f"\nsilver/epe_consumo.csv pronto — {len(epe):,} linhas")

25 combinacoes regiao x classe, 132 meses cada -> OK

silver/epe_consumo.csv pronto — 32,876 linhas


CADE O DICIONARIO

In [23]:
epe

,data,uf,regiao,sistema,sistema_isolado,classe,tipo_consumidor,consumo_mwh,consumidores,consumo_por_consumidor
0,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Comercial,Cativo,2101.000,4376,0.480119
1,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Industrial,Cativo,274.000,172,1.593023
2,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Outros,Cativo,3413.000,1248,2.734776
3,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Residencial,Cativo,6497.000,48170,0.134876
4,2015-01-01,AC,Norte,SISTEMAS ISOLADOS,True,Rural,Cativo,1141.000,5826,0.195846
...,...,...,...,...,...,...,...,...,...,...
32871,2025-12-01,TO,Norte,NORTE INTERLIGADO,False,Outros,Cativo,27948.834,10540,2.651692
32872,2025-12-01,TO,Norte,NORTE INTERLIGADO,False,Outros,Livre,4962.729,98,50.640092
32873,2025-12-01,TO,Norte,NORTE INTERLIGADO,False,Residencial,Cativo,131987.972,616169,0.214207
32874,2025-12-01,TO,Norte,NORTE INTERLIGADO,False,Rural,Cativo,15680.250,38815,0.403974
